# Protein Aggregation, Solubility, Structural, and Biophysical Measurements for Escherichia coli Cytoplasmic Proteins in Macromolecular Crowding Conditions Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.59w0-0ytm/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list the available `RecordSet` objects in the dataset and inspect their fields and column `@id`s.

In [ ]:
print("Available Record Sets (by @id):")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"  - {rs.id}: {rs.name}")
    record_sets.append(rs.id)

print("\nFields and Columns for each Record Set:")
for rs in dataset.metadata.record_sets:
    print(f"\n[RecordSet] {rs.id} : {rs.name}")
    for field in rs.fields:
        print(f"    Field: {field.id} | {field.name} | Data type: {field.data_type}")
        # Show columns if present
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        Column: {col.id} | {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract all record sets
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}")
    print(df.head(2), '\n')

# For the rest of the notebook, pick the first record set for demonstration
demo_record_set_id = record_sets[0] if record_sets else None
print(f"Demo Record Set ID: {demo_record_set_id}")
print("Available columns in the demo DataFrame:")
print(dataframes[demo_record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, we select the first numeric field detected in the DataFrame using its `@id`. This demonstrates how to programmatically select and use IDs for analyses.

In [ ]:
import numpy as np

# Pick a numeric field from the first record set
numeric_field_id = None
group_field_id = None

df = dataframes[demo_record_set_id]

# Try to find a numeric field
for rs in dataset.metadata.record_sets:
    if rs.id == demo_record_set_id:
        for field in rs.fields:
            # Typical numeric data types
            if field.data_type in ['Float', 'Integer', 'Number']:
                if field.id in df.columns:
                    numeric_field_id = field.id
                    break
        # Try to find a group field (categorical)
        for field in rs.fields:
            if field.data_type in ['Text', 'String', 'Categorical'] and field.id in df.columns:
                group_field_id = field.id
                break
        break

if numeric_field_id is None:
    raise ValueError('No numeric field found in the first record set!')
print(f"Using numeric field @id: {numeric_field_id}")
threshold = df[numeric_field_id].quantile(0.25) if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0

# Filter: keep rows where field > threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head(3))

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by first available group/categorical field if present
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouped, plot group means
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset metadata and data using the Croissant JSON-LD schema and `mlcroissant`.
- All operations reference entities by their `@id` for clarity and reproducibility.
- Numeric fields were filtered and normalized; basic visualizations help understand the data distribution.
- For additional analysis, inspect further fields and record sets using their `@id`s.